# Classificação de sujeira em placas solares — Transfer Learning (ResNet) · dataset tratado

Treina a classificação **NaoLimpar × Limpar** com uma ResNet pré-treinada no ImageNet, usando o dataset já
tratado (224×224, enquadrado, cores originais) pelo notebook de tratamento.

| Etapa | O que faz |
|---|---|
| 3–4. Dados | lê o dataset tratado e **separa o teste** (ou usa um teste externo); agrupa cada arquivo pela foto original (cópias do Roboflow e sintéticas `synth__` ficam juntas) e **confere se nenhuma foto do teste está no treino** |
| 6. Aumento de dados | **rotações de 90° (0/90/180/270)** + **ângulo aleatório de ±N graus** (bordas por reflexo, sem cantos vazios) + espelhamento; só no treino |
| 9. Validação cruzada | **quantidade de folds ajustável** (`N_FOLDS_TO_RUN`). Sem validação durante o treino: a cada época, **acurácia e loss do treino**; no fim de cada fold, **acurácia e loss do teste do fold** |
| 10. Treino final | **do zero**, com **todo o dataset de treino** (tudo menos o teste) |
| 11. Métricas do modelo final | no **teste**: acurácia, loss, precisão/recall/F1, AUC, matriz de confusão, curvas, intervalos de confiança |
| 12–13. Erros e exportação | Grad-CAM dos erros, `model.onnx`, `labels.json`, `model_card.json`, `.pth` |

> Rode no Colab com GPU: *Ambiente de execução → Alterar tipo → GPU*. Ajuste só a célula 2.

## 1. Imports

In [ ]:
import copy, hashlib, json, math, os, random, re, shutil, sys, time, zipfile
import datetime as dt
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IN_COLAB = "google.colab" in sys.modules
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", DEVICE, "| Colab:", IN_COLAB)
if DEVICE.type != "cuda":
    print("ATENÇÃO: sem GPU o treino da ResNet fica muito lento. No Colab: Ambiente de execução → Alterar tipo → GPU.")

## 2. Configuração (o único lugar que você precisa mexer)

- `TRAIN_PATH`: zip **ou** pasta do dataset tratado (224×224), com as subpastas `NaoLimpar` e `Limpar`.
- `TEST_PATH = None` *(padrão)*: o notebook **separa o teste** com `TEST_FRAC` das fotos **reais**, por foto original
  (cópias do Roboflow e sintéticas da mesma foto ficam do mesmo lado). Para usar um teste já separado, informe o caminho.
- Se o arquivo não for encontrado, o notebook procura pelo mesmo nome em `/content` e no Drive; se ainda assim não
  achar, pede o upload. O zip enviado é extraído em `WORK_DIR` e, com `SAVE_UPLOADS_TO_DRIVE=True`, copiado para
  `DRIVE_DIR`: nas próximas execuções ele é reaproveitado sem pedir de novo.
- `N_FOLDS_TO_RUN`: quantos folds da validação cruzada rodar (`0` pula a validação cruzada e vai direto ao treino
  final; `N_SPLITS` roda todos).
- `THRESHOLD_MODE`: como escolher o limiar de decisão a partir dos folds (`"f1"`, `"recall"` ou `"fixo"` = 0,5).

In [ ]:
CFG = dict(
    # ---------- dados (já tratados: 224x224, enquadrados) ----------
    TRAIN_PATH = "/content/Datasetv6.zip"      if IN_COLAB else r"C:\Users\Aluno\Downloads\Datasetv6.zip",
    TEST_PATH  = None,                       # None = separa o teste dentro do notebook | ou zip/pasta de um teste já separado
    TEST_FRAC  = 0.15,                       # (com TEST_PATH=None) fração das fotos REAIS que vira teste
    WORK_DIR   = "/content/dados"                  if IN_COLAB else r"C:\Users\Aluno\Downloads\dados",
    SAVE_UPLOADS_TO_DRIVE = True,            # Colab: guarda no Drive os zips enviados por upload (não precisa reenviar)
    DRIVE_DIR  = "/content/drive/MyDrive/placas_dados",
    OUT_DIR    = "saida_placas_resnet",
    NEG_CLASS  = "NaoLimpar",                # pasta da classe que NÃO aciona a limpeza
    POS_CLASS  = "Limpar",                   # pasta da classe que ACIONA a limpeza
    SYNTH_PREFIXES = ("synth__", "civitai__"),   # imagens geradas: nunca entram no teste dos folds
    EXCLUDE_FILE = "excluir.txt",            # nomes de arquivos a ignorar (um por linha); opcional
    IMG_SIZE   = 224,

    # ---------- modelo ----------
    ARCH          = "resnet50",              # "resnet18" | "resnet34" | "resnet50"
    PRETRAINED    = True,
    DROPOUT       = 0.3,
    FREEZE_EPOCHS = 2,                       # épocas iniciais só com a cabeça treinando (0 = tudo desde o início)

    # ---------- treino ----------
    EPOCHS        = 25,
    BATCH         = 32,
    LR            = 3e-4,
    WEIGHT_DECAY  = 1e-4,
    LABEL_SMOOTHING = 0.05,
    PRINT_EVERY   = 1,

    # ---------- aumento de dados (só no treino) ----------
    AUG_ROT90      = True,                   # sorteia 0/90/180/270 graus
    AUG_RANDOM_DEG = 10,                     # + ângulo aleatório de -N a +N graus (0 = desliga)
    AUG_FLIP       = True,                   # espelhamento horizontal
    COLOR_JITTER   = False,                  # brilho/contraste leves (a cor da sujeira é informação)

    # ---------- validação cruzada ----------
    N_SPLITS       = 5,
    N_FOLDS_TO_RUN = 2,                      # 0 = pula | 1..N_SPLITS = quantos folds rodar
    THRESHOLD_MODE = "f1",                   # "f1" | "recall" | "fixo" (0,5)
    RECALL_TARGET  = 0.90,                   # usado com THRESHOLD_MODE = "recall"

    # ---------- avaliação / exportação ----------
    N_BOOTSTRAP    = 1000,
    N_ERRORS_SHOW  = 18,
    EXPORT_ONNX    = True,
    ONNX_OPSET     = 17,

    NUM_WORKERS = 0 if os.name == "nt" else 2,
    SEED        = 42,
    QUICK       = False,                     # True = teste rápido (1 época, 1 fold, imagens 128)
)
if CFG["QUICK"]:
    CFG.update(EPOCHS=1, N_FOLDS_TO_RUN=1, IMG_SIZE=128, N_BOOTSTRAP=100, NUM_WORKERS=0)
assert CFG["ARCH"] in ("resnet18", "resnet34", "resnet50")
assert 0 <= CFG["N_FOLDS_TO_RUN"] <= CFG["N_SPLITS"]
assert CFG["THRESHOLD_MODE"] in ("f1", "recall", "fixo")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(CFG["SEED"])
OUT = Path(CFG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG["IMG_SIZE"]
CLASS_NAMES = [CFG["NEG_CLASS"], CFG["POS_CLASS"]]     # 0 = NaoLimpar, 1 = Limpar
POS = 1
CFG

## 3. Carregar o dataset e separar treino e teste

Com `TEST_PATH = None`, o teste sai do próprio dataset **antes de qualquer treino**: `TEST_FRAC` das fotos reais,
estratificado por classe e **por foto original**. Imagens sintéticas nunca vão para o teste, e as geradas a partir
de uma foto do teste saem do treino. A divisão fica salva em `particao.csv`.

Cada imagem é lida em RGB. Se alguma não estiver em `IMG_SIZE × IMG_SIZE` (por exemplo, não passou pelo
tratamento), ela é redimensionada e o notebook avisa.

Os dados extraídos ficam em `WORK_DIR/treino` e `WORK_DIR/teste`. Para trocar de dataset, apague essas pastas
(ou mude `WORK_DIR`) antes de rodar de novo.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _has_images(folder):
    return folder.exists() and any(p.suffix.lower() in IMG_EXTS for p in folder.rglob("*") if p.is_file())

def get_root(key, name):
    # devolve a pasta com as imagens de `name` ("treino" ou "teste"), extraindo/enviando o zip se preciso
    dest = Path(CFG["WORK_DIR"]) / name
    if _has_images(dest):                                    # já extraído antes (nesta sessão ou no disco local)
        print(f"[{name}] usando os dados já extraídos em {dest}")
        return dest
    path = Path(CFG[key])
    if not path.exists():                                    # procura pelo mesmo nome em /content e no Drive
        cands = [Path("/content") / path.name, Path(CFG["DRIVE_DIR"]) / path.name, Path("/content/drive/MyDrive") / path.name]
        path = next((c for c in cands if c.exists()), path)
    if not path.exists() and IN_COLAB and CFG["SAVE_UPLOADS_TO_DRIVE"]:
        try:                                                 # pode estar no Drive de uma execução anterior
            from google.colab import drive
            drive.mount("/content/drive")
            salvos = sorted(Path(CFG["DRIVE_DIR"]).glob(f"{name}__*.zip")) if Path(CFG["DRIVE_DIR"]).exists() else []
            if salvos:
                path = salvos[-1]
        except Exception as e:
            print("não consegui montar o Drive:", e)
    if not path.exists():
        if not IN_COLAB:
            raise FileNotFoundError(f"[{name}] não encontrado: {CFG[key]}. Ajuste {key} na célula 2.")
        from google.colab import files
        print(f"[{name}] {CFG[key]} não encontrado. Selecione o ZIP de {name} (com as pastas {CLASS_NAMES}):")
        up = files.upload()
        if not up:
            raise RuntimeError(f"[{name}] nenhum arquivo enviado.")
        fname = next(iter(up))
        path = Path("/content") / fname
        if not path.exists():                                # garante que o arquivo enviado foi gravado em disco
            path.write_bytes(up[fname])
        if path.suffix.lower() != ".zip":
            raise ValueError(f"[{name}] envie um arquivo .zip (recebi {fname}). Compacte a pasta com as subpastas das classes.")
        if CFG["SAVE_UPLOADS_TO_DRIVE"]:
            try:
                from google.colab import drive
                drive.mount("/content/drive")
                Path(CFG["DRIVE_DIR"]).mkdir(parents=True, exist_ok=True)
                copia = Path(CFG["DRIVE_DIR"]) / f"{name}__{fname}"
                shutil.copy(path, copia)
                print(f"[{name}] cópia salva no Drive: {copia}")
            except Exception as e:
                print(f"[{name}] não consegui salvar no Drive ({e}); o arquivo fica só nesta sessão.")
    CFG[key] = str(path)                                     # lembra o caminho certo para o resto do notebook
    if path.is_dir():
        if not _has_images(path):
            raise FileNotFoundError(f"[{name}] a pasta {path} não tem imagens.")
        print(f"[{name}] usando a pasta {path}")
        return path
    if path.suffix.lower() != ".zip":
        raise ValueError(f"[{name}] {path} não é pasta nem .zip.")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(dest)
    if not _has_images(dest):
        raise RuntimeError(f"[{name}] o zip {path.name} não tem imagens.")
    print(f"[{name}] {path.name} extraído em {dest}")
    return dest

def read_rgb(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    return None if img is None else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def source_id(path):
    s = Path(path).stem
    if s.lower().startswith("synth__"):
        s = re.sub(r"__s\d+$", "", s[len("synth__"):])
        s = s.rsplit("__", 1)[0] if "__" in s else s
    s = re.sub(r"^(train|valid|test)_+", "", s, flags=re.I)
    s = re.sub(r"\.rf\.[0-9a-f]+$", "", s, flags=re.I)
    s = re.sub(r"_(jpe?g|png|bmp|webp)(__\w+)?$", "", s, flags=re.I)
    return s.lower()

def source_family(path):
    return re.sub(r"[\s_\-\(\)]*\d.*$", "", source_id(path)) or "outros"

def load_set(root, name):
    found = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS and "__MACOSX" not in p.parts:
            found.setdefault(p.parent.name, []).append(p)
    faltando = [c for c in CLASS_NAMES if c not in found]
    assert not faltando, f"[{name}] pastas {faltando} não encontradas em {root} (achei {sorted(found)})"
    excl = set()
    ex = Path(CFG["EXCLUDE_FILE"]) if CFG["EXCLUDE_FILE"] else None
    if ex and ex.exists():
        excl = {l.strip() for l in ex.read_text(encoding="utf-8").splitlines() if l.strip()}
    rows, imgs, redim, vistos, dup = [], [], 0, set(), 0
    for c in CLASS_NAMES:
        for p in sorted(found[c]):
            if p.name in excl:
                continue
            raw = p.read_bytes()
            h = hashlib.md5(raw).hexdigest()
            if h in vistos:
                dup += 1; continue
            img = read_rgb(p)
            if img is None:
                print(f"[{name}] ilegível: {p.name}"); continue
            if img.shape[:2] != (S, S):
                redim += 1
                img = cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA if max(img.shape[:2]) > S else cv2.INTER_CUBIC)
            vistos.add(h)
            imgs.append(img)
            rows.append(dict(file=str(p), arquivo=p.name, classe=c, label=CLASS_NAMES.index(c), md5=h,
                             source=source_id(p), fonte=source_family(p),
                             sintetica=p.name.lower().startswith(tuple(x.lower() for x in CFG["SYNTH_PREFIXES"]))))
    meta = pd.DataFrame(rows)
    X = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).contiguous()
    print(f"[{name}] {len(meta)} imagens | duplicatas ignoradas: {dup} | redimensionadas: {redim}"
          + (" (ATENÇÃO: passe este conjunto pelo notebook de tratamento)" if redim > 0.1 * max(len(meta), 1) else ""))
    return meta, X

meta_all, X_all = load_set(get_root("TRAIN_PATH", "treino"), "dataset")

if CFG["TEST_PATH"]:
    meta, X = meta_all, X_all
    meta_te, X_te = load_set(get_root("TEST_PATH", "teste"), "teste")
    print("teste externo:", CFG["TEST_PATH"])
else:
    real = np.where(~meta_all.sintetica.to_numpy())[0]
    k = max(2, round(1 / CFG["TEST_FRAC"]))
    sg = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=CFG["SEED"])
    _, te_pos = next(sg.split(np.zeros(len(real)), meta_all.label.to_numpy()[real], meta_all.source.to_numpy()[real]))
    te_idx = real[te_pos]
    g_te = set(meta_all.source.to_numpy()[te_idx])
    te_set = set(te_idx.tolist())
    tr_idx = np.array([i for i in range(len(meta_all)) if i not in te_set and meta_all.source[i] not in g_te], dtype=int)
    descartadas = len(meta_all) - len(te_idx) - len(tr_idx)
    meta, X = meta_all.iloc[tr_idx].reset_index(drop=True), X_all[torch.as_tensor(tr_idx)]
    meta_te, X_te = meta_all.iloc[te_idx].reset_index(drop=True), X_all[torch.as_tensor(te_idx)]
    print(f"split interno: treino {len(meta)} | teste {len(meta_te)} ({len(meta_te) / max(len(real), 1):.1%} das fotos reais)"
          + (f" | {descartadas} sintéticas descartadas (geradas de fotos do teste)" if descartadas else ""))
del X_all
pd.concat([meta.assign(particao="treino"), meta_te.assign(particao="teste")])[
    ["arquivo", "classe", "particao", "source", "sintetica"]].to_csv(OUT / "particao.csv", index=False)
print("divisão salva em", OUT / "particao.csv")

y, y_te = meta.label.to_numpy(), meta_te.label.to_numpy()
groups = meta.source.to_numpy()
print(f"\ntreino: {X.numel() / 1e6:.0f} MB em memória | teste: {X_te.numel() / 1e6:.0f} MB")

## 4. Conferências: equilíbrio, sintéticas e vazamento treino → teste

O teste só mede o modelo de verdade se **nenhuma** foto dele estiver no treino. A célula procura:
- arquivos **idênticos** (mesmo conteúdo) nos dois conjuntos;
- a **mesma foto original** (cópias do Roboflow ou sintéticas geradas a partir dela) nos dois conjuntos.

In [ ]:
tab = pd.DataFrame({
    "treino (reais)": meta[~meta.sintetica].classe.value_counts(),
    "treino (sintéticas)": meta[meta.sintetica].classe.value_counts(),
    "teste": meta_te.classe.value_counts(),
}).reindex(CLASS_NAMES).fillna(0).astype(int)
display(tab)

iguais = set(meta.md5) & set(meta_te.md5)
mesma_foto = set(meta.source) & set(meta_te.source)
print(f"arquivos idênticos em treino e teste: {len(iguais)}")
print(f"mesma foto original em treino e teste: {len(mesma_foto)}")
if iguais or mesma_foto:
    lista = sorted(mesma_foto)[:15]
    print("  exemplos:", lista)
    print("ATENÇÃO: as métricas do teste ficarão otimistas. Remova essas fotos do treino (ou do teste) e rode de novo.")
if (meta_te.sintetica).any():
    print(f"ATENÇÃO: {meta_te.sintetica.sum()} imagens sintéticas no TESTE. O teste deve ter só fotos reais.")

fig, ax = plt.subplots(1, 2, figsize=(14, 3.4))
tab.plot(kind="bar", ax=ax[0], rot=0); ax[0].set_title("Imagens por classe")
fonte = pd.crosstab(meta.fonte, meta.classe).reindex(columns=CLASS_NAMES, fill_value=0)
fonte.loc[fonte.sum(1).sort_values().index[-12:]].plot(kind="barh", stacked=True, ax=ax[1])
ax[1].set_title("Treino: fontes mais frequentes × classe"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(15, 4.2))
rng = np.random.default_rng(CFG["SEED"])
for c in range(2):
    for j, i in enumerate(rng.choice(np.where(y == c)[0], min(8, (y == c).sum()), replace=False)):
        axes[c, j].imshow(X[i].permute(1, 2, 0).numpy())
    for j in range(8):
        axes[c, j].axis("off")
    axes[c, 0].set_title(CLASS_NAMES[c], loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 5. Folds da validação cruzada

`StratifiedGroupKFold` sobre as fotos **reais** do treino: todas as versões da mesma foto original ficam no
mesmo lado. As sintéticas entram só na parte de treino de cada fold (e saem se a foto original estiver no
teste do fold).

In [ ]:
idx_real = np.where(~meta.sintetica.to_numpy())[0]
idx_syn = np.where(meta.sintetica.to_numpy())[0]
ALL_IDX = np.arange(len(y))

def dist(yy):
    c = np.bincount(yy, minlength=2)
    return " | ".join(f"{n}: {c[k]:5d} ({c[k] / max(len(yy), 1):5.1%})" for k, n in enumerate(CLASS_NAMES))

FOLDS = []
if CFG["N_FOLDS_TO_RUN"] > 0:
    skf = StratifiedGroupKFold(n_splits=CFG["N_SPLITS"], shuffle=True, random_state=CFG["SEED"])
    for tr, va in skf.split(np.zeros(len(idx_real)), y[idx_real], groups[idx_real]):
        te_idx = idx_real[va]
        g_te = set(groups[te_idx])
        tr_idx = np.concatenate([idx_real[tr], [i for i in idx_syn if groups[i] not in g_te]]).astype(int)
        FOLDS.append((tr_idx, te_idx))
    for k, (tr, te) in enumerate(FOLDS, 1):
        marca = "  <- roda" if k <= CFG["N_FOLDS_TO_RUN"] else ""
        print(f"fold {k}: treino {len(tr):5d} | teste {len(te):5d}: {dist(y[te])}{marca}")
else:
    print("N_FOLDS_TO_RUN = 0: validação cruzada desativada")

## 6. Aumento de dados (só no treino)

- **Rotações de 90°** (`AUG_ROT90`): 0°, 90°, 180° ou 270°, sorteado a cada época — exatas, sem perda.
- **Ângulo aleatório** (`AUG_RANDOM_DEG`): a imagem é estendida por reflexo, girada e recortada no centro,
  então não aparecem cantos vazios.
- **Espelhamento** (`AUG_FLIP`) e, opcionalmente, brilho/contraste leves.

Iguais para as duas classes. As imagens de teste nunca recebem aumento.

In [ ]:
class RandomRot90:
    def __call__(self, x):
        return torch.rot90(x, random.randint(0, 3), dims=(1, 2))

class RandomRotateReflect:
    def __init__(self, degrees, p=0.7):
        self.degrees, self.p = degrees, p
    def __call__(self, x):
        if self.degrees <= 0 or random.random() > self.p:
            return x
        size = x.shape[-1]
        pad = int(math.ceil(size * 0.21))
        xf = F.pad(x.float().unsqueeze(0), [pad] * 4, mode="reflect").squeeze(0)
        xf = TF.rotate(xf, random.uniform(-self.degrees, self.degrees), interpolation=T.InterpolationMode.BILINEAR)
        return TF.center_crop(xf, [size, size]).round().clamp(0, 255).to(torch.uint8)

aug = []
if CFG["AUG_ROT90"]:
    aug.append(RandomRot90())
if CFG["AUG_FLIP"]:
    aug.append(T.RandomHorizontalFlip())
if CFG["AUG_RANDOM_DEG"] > 0:
    aug.append(RandomRotateReflect(CFG["AUG_RANDOM_DEG"]))
if CFG["COLOR_JITTER"]:
    aug.append(T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1))
tf_train = T.Compose(aug) if aug else None

class ImgDataset(Dataset):
    def __init__(self, Xs, ys, idx, tf=None):
        self.Xs, self.ys, self.idx, self.tf = Xs, ys, np.asarray(idx, dtype=int), tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = self.Xs[j]
        if self.tf is not None:
            img = self.tf(img)
        return img, int(self.ys[j])

_MEAN = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
_STD = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
def to_input(xb):
    xb = xb.to(DEVICE, non_blocking=True).float().div_(255.0)
    return (xb - _MEAN) / _STD

PIN = DEVICE.type == "cuda"
def make_loader(Xs, ys, idx, train):
    ds = ImgDataset(Xs, ys, idx, tf_train if train else None)
    if train:
        return DataLoader(ds, batch_size=CFG["BATCH"], shuffle=True, drop_last=len(idx) > CFG["BATCH"],
                          num_workers=CFG["NUM_WORKERS"], pin_memory=PIN)
    return DataLoader(ds, batch_size=CFG["BATCH"] * 2, shuffle=False, num_workers=CFG["NUM_WORKERS"], pin_memory=PIN)

fig, axes = plt.subplots(2, 7, figsize=(15, 4.6))
for c in range(2):
    i = np.where(y == c)[0][0]
    axes[c, 0].imshow(X[i].permute(1, 2, 0).numpy()); axes[c, 0].set_title(f"{CLASS_NAMES[c]} original", fontsize=8)
    for j in range(1, 7):
        axes[c, j].imshow((tf_train(X[i]) if tf_train else X[i]).permute(1, 2, 0).numpy())
        axes[c, j].set_title("aumentada", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Aumento de dados: rotações de 90°, ângulo aleatório (com reflexo) e espelhamento")
plt.tight_layout(); plt.show()

## 7. Modelo: ResNet pré-treinada (transfer learning)

Pesos do ImageNet e a última camada trocada por `Dropout + Linear(2)`. Nas primeiras `FREEZE_EPOCHS` épocas só a
cabeça treina; depois, a rede inteira (com o mesmo OneCycleLR).

In [ ]:
def build_model(arch=CFG["ARCH"], pretrained=CFG["PRETRAINED"]):
    ctor = getattr(torchvision.models, arch)
    m = None
    if pretrained:
        try:
            m = ctor(weights=torchvision.models.get_model_weights(arch).DEFAULT)
        except Exception as e:
            print("sem acesso aos pesos pré-treinados, iniciando do zero:", type(e).__name__)
    if m is None:
        m = ctor(weights=None)
    m.fc = nn.Sequential(nn.Dropout(CFG["DROPOUT"]), nn.Linear(m.fc.in_features, 2))
    return m

def set_backbone_trainable(model, flag):
    for n, p in model.named_parameters():
        if not n.startswith("fc."):
            p.requires_grad = flag

_m = build_model()
print(f"{CFG['ARCH']}: {sum(p.numel() for p in _m.parameters()) / 1e6:.1f} M parâmetros")
del _m

## 8. Funções de treino e avaliação

Treino **sem validação**: a cada época aparecem só a **acurácia e a loss do treino**. O conjunto de teste é
avaliado uma única vez, no fim. A loss de teste é a entropia cruzada comum (sem pesos e sem label smoothing),
para ser comparável entre execuções.

In [ ]:
CE = nn.CrossEntropyLoss()

def train_model(Xs, ys, train_idx, name, seed=CFG["SEED"]):
    seed_all(seed)
    counts = np.bincount(ys[train_idx], minlength=2).astype(float)
    w = torch.tensor(counts.sum() / (2 * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["LABEL_SMOOTHING"])
    model = build_model().to(DEVICE)
    loader = make_loader(Xs, ys, train_idx, train=True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["LR"], epochs=CFG["EPOCHS"],
                                                steps_per_epoch=max(1, len(loader)), pct_start=0.2)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    print(f"[{name}] {len(train_idx)} imagens de treino ({dist(ys[train_idx])})")
    hist, t_all = [], time.time()
    for ep in range(1, CFG["EPOCHS"] + 1):
        t0 = time.time()
        if CFG["FREEZE_EPOCHS"]:
            set_backbone_trainable(model, ep > CFG["FREEZE_EPOCHS"])
        model.train()
        tot, correct, n = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = to_input(xb), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                out = model(xb)
                loss = criterion(out, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb); correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
        hist.append(dict(epoca=ep, loss_treino=tot / max(n, 1), acc_treino=correct / max(n, 1)))
        if ep == 1 or ep % CFG["PRINT_EVERY"] == 0 or ep == CFG["EPOCHS"]:
            print(f"[{name}] época {ep:3d}/{CFG['EPOCHS']} | loss treino {hist[-1]['loss_treino']:.4f} | "
                  f"acc treino {hist[-1]['acc_treino'] * 100:6.2f}% | {time.time() - t0:.1f}s")
    print(f"[{name}] tempo: {(time.time() - t_all) / 60:.1f} min")
    return model.eval(), pd.DataFrame(hist)

@torch.no_grad()
def evaluate(model, Xs, ys, idx):
    # devolve (P(Limpar) de cada imagem, loss média, acurácia no corte 0,5)
    model.eval()
    probs, losses = [], []
    for xb, yb in make_loader(Xs, ys, idx, train=False):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(to_input(xb)).float()
        yb = yb.to(DEVICE)
        losses.append(CE(logits, yb).item() * len(yb))
        probs.append(logits.softmax(1)[:, POS].cpu())
    p = torch.cat(probs).numpy()
    return p, sum(losses) / len(idx), accuracy_score(ys[idx], (p >= 0.5).astype(int))

def choose_threshold(yy, p):
    if CFG["THRESHOLD_MODE"] == "fixo":
        return 0.5
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    if CFG["THRESHOLD_MODE"] == "recall":
        ok = [t for t in grid if recall_score(yy, (p >= t).astype(int), zero_division=0) >= CFG["RECALL_TARGET"]]
        return float(max(ok)) if ok else 0.5
    return float(grid[int(np.argmax([f1_score(yy, (p >= t).astype(int), average="macro") for t in grid]))])

def free_gpu():
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def plot_train(hists, titles, test_marks=None):
    fig, axes = plt.subplots(1, len(hists), figsize=(4.8 * len(hists), 3.4), squeeze=False)
    for k, (ax, h, t) in enumerate(zip(axes[0], hists, titles)):
        ax.plot(h.epoca, h.loss_treino, "o-", ms=3, c="#c53030", label="loss treino")
        ax.set_xlabel("época"); ax.set_ylabel("loss"); ax.set_title(t, fontsize=9)
        ax2 = ax.twinx()
        ax2.plot(h.epoca, h.acc_treino * 100, "o-", ms=3, c="#2b6cb0", label="acc treino")
        if test_marks:
            lt, at = test_marks[k]
            ax.scatter([h.epoca.iloc[-1]], [lt], marker="s", s=60, c="#c53030", edgecolor="k", zorder=5, label="loss teste")
            ax2.scatter([h.epoca.iloc[-1]], [at * 100], marker="s", s=60, c="#2b6cb0", edgecolor="k", zorder=5, label="acc teste")
        ax2.set_ylim(0, 100); ax2.set_ylabel("acurácia (%)"); ax2.grid(alpha=0.3)
        if k == 0:
            h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
            fig.legend(h1 + h2, l1 + l2, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.1))
    plt.tight_layout(); plt.show()

## 9. Validação cruzada (folds ajustáveis)

Roda os primeiros `N_FOLDS_TO_RUN` folds. Cada fold treina do zero e, no fim, mostra **acurácia e loss do
treino e do teste do fold**. As previsões dos folds escolhem o **limiar de decisão** usado no modelo final.

In [ ]:
THRESHOLD = 0.5
oof = np.full(len(y), np.nan)
cv_rows, cv_hists = [], []
for k, (tr, te) in enumerate(FOLDS[:CFG["N_FOLDS_TO_RUN"]], 1):
    print(f"\n{'=' * 80}\nFOLD {k}/{CFG['N_SPLITS']}\n{'=' * 80}")
    model, hist = train_model(X, y, tr, f"fold {k}", seed=CFG["SEED"] + k)
    p, loss_te, acc_te = evaluate(model, X, y, te)
    oof[te] = p
    row = dict(fold=k, acc_treino=hist.acc_treino.iloc[-1], loss_treino=hist.loss_treino.iloc[-1],
               acc_teste=acc_te, loss_teste=loss_te, f1_macro_teste=f1_score(y[te], (p >= 0.5).astype(int), average="macro"))
    cv_rows.append(row); cv_hists.append(hist)
    print(f"RESULTADO fold {k}: treino acc {row['acc_treino'] * 100:.2f}% loss {row['loss_treino']:.4f} | "
          f"teste acc {acc_te * 100:.2f}% loss {loss_te:.4f}")
    del model; free_gpu()

if cv_rows:
    df_cv = pd.DataFrame(cv_rows)
    resumo = df_cv[["acc_treino", "loss_treino", "acc_teste", "loss_teste", "f1_macro_teste"]].agg(["mean", "std"])
    display(pd.concat([df_cv.set_index("fold"), resumo]).round(4))
    plot_train(cv_hists, [f"fold {k}" for k in df_cv.fold], [(r["loss_teste"], r["acc_teste"]) for r in cv_rows])
    m = ~np.isnan(oof)
    THRESHOLD = choose_threshold(y[m], oof[m])
    print(f"limiar de decisão ({CFG['THRESHOLD_MODE']}): {THRESHOLD:.2f}")
    gap = (df_cv.acc_treino - df_cv.acc_teste).mean()
    if gap > 0.10:
        print(f"Treino {gap * 100:.0f} pontos acima do teste: OVERFITTING. Reduza EPOCHS, aumente DROPOUT ou use resnet18.")
else:
    df_cv = None
    print("validação cruzada pulada: limiar fixo em 0,50")

## 10. Treino final (do zero, com todo o dataset de treino)

Um modelo **novo**, a partir dos pesos do ImageNet, treinado com **todas** as imagens de treino (reais +
sintéticas). Nenhum peso dos folds é reaproveitado, e as imagens de teste continuam fora do treino.

In [ ]:
final_model, hist_final = train_model(X, y, ALL_IDX, "modelo final", seed=CFG["SEED"])
torch.save(final_model.state_dict(), OUT / "modelo_placas.pth")
print("salvo:", OUT / "modelo_placas.pth")

## 11. Métricas de avaliação do modelo final (teste)

- acurácia e **loss** de treino e de teste;
- acurácia balanceada, precisão / recall / F1 por classe, F1-macro, AUC ROC e AUC PR;
- matriz de confusão no limiar escolhido (e no corte 0,5);
- **intervalos de confiança de 95%** por bootstrap;
- curvas ROC e precisão-recall, distribuição das probabilidades e desempenho por fonte.

Salvo em `metricas_modelo_final.json` e `metricas_teste.png`.

In [ ]:
TE_IDX = np.arange(len(y_te))
p_te, loss_te, acc_te_05 = evaluate(final_model, X_te, y_te, TE_IDX)
_, loss_tr_eval, acc_tr_eval = evaluate(final_model, X, y, ALL_IDX)     # treino sem aumento, só para comparar

def metricas(yy, p, thr):
    pred = (p >= thr).astype(int)
    cm = confusion_matrix(yy, pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
    dois = len(set(yy)) == 2
    return {"limiar": thr, "n": int(len(yy)),
            "acuracia": accuracy_score(yy, pred), "acuracia_balanceada": balanced_accuracy_score(yy, pred),
            "f1_macro": f1_score(yy, pred, average="macro"),
            f"precisao_{CLASS_NAMES[1]}": precision_score(yy, pred, zero_division=0),
            f"recall_{CLASS_NAMES[1]}": recall_score(yy, pred, zero_division=0),
            f"f1_{CLASS_NAMES[1]}": f1_score(yy, pred, zero_division=0),
            f"precisao_{CLASS_NAMES[0]}": tn / max(tn + fn, 1),
            f"recall_{CLASS_NAMES[0]}": tn / max(tn + fp, 1),
            f"f1_{CLASS_NAMES[0]}": f1_score(1 - yy, 1 - pred, zero_division=0),
            "auc_roc": roc_auc_score(yy, p) if dois else float("nan"),
            "auc_pr": average_precision_score(yy, p) if dois else float("nan"),
            "limpas_que_acionariam_pct": fp / max(tn + fp, 1), "sujas_ignoradas_pct": fn / max(fn + tp, 1),
            "matriz_confusao": cm.tolist()}

def bootstrap(yy, p, thr, n=CFG["N_BOOTSTRAP"]):
    rng = np.random.default_rng(CFG["SEED"]); a, f, u = [], [], []
    for _ in range(n):
        i = rng.integers(0, len(yy), len(yy))
        if len(set(yy[i])) < 2:
            continue
        pr = (p[i] >= thr).astype(int)
        a.append(accuracy_score(yy[i], pr)); f.append(f1_score(yy[i], pr, average="macro")); u.append(roc_auc_score(yy[i], p[i]))
    q = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
    return {"acuracia": q(a), "f1_macro": q(f), "auc_roc": q(u)}

M = metricas(y_te, p_te, THRESHOLD)
M05 = metricas(y_te, p_te, 0.5)
IC = bootstrap(y_te, p_te, THRESHOLD)

print("=" * 80)
print(f"MODELO FINAL · {CFG['ARCH']} · teste com {len(y_te)} imagens nunca vistas · limiar {THRESHOLD:.2f}")
print("=" * 80)
print(f"{'':22s}{'acurácia':>12s}{'loss':>10s}")
print(f"{'treino (última época)':22s}{hist_final.acc_treino.iloc[-1] * 100:11.2f}%{hist_final.loss_treino.iloc[-1]:10.4f}   (com aumento de dados)")
print(f"{'treino (sem aumento)':22s}{acc_tr_eval * 100:11.2f}%{loss_tr_eval:10.4f}")
print(f"{'teste':22s}{acc_te_05 * 100:11.2f}%{loss_te:10.4f}   (corte 0,50)")
print("-" * 80)
print(f"Acurácia (limiar {THRESHOLD:.2f}) : {M['acuracia']:.4f}  IC95% {IC['acuracia'][0]:.3f}–{IC['acuracia'][1]:.3f}")
print(f"Acurácia balanceada     : {M['acuracia_balanceada']:.4f}")
print(f"F1-macro                : {M['f1_macro']:.4f}  IC95% {IC['f1_macro'][0]:.3f}–{IC['f1_macro'][1]:.3f}")
print(f"AUC ROC                 : {M['auc_roc']:.4f}  IC95% {IC['auc_roc'][0]:.3f}–{IC['auc_roc'][1]:.3f}")
print(f"AUC PR                  : {M['auc_pr']:.4f}")
for c in CLASS_NAMES[::-1]:
    print(f"{c:10s} precisão {M[f'precisao_{c}']:.4f} | recall {M[f'recall_{c}']:.4f} | F1 {M[f'f1_{c}']:.4f}")
print(f"Placas limpas que acionariam a limpeza: {M['limpas_que_acionariam_pct']:.1%}")
print(f"Placas sujas que seriam ignoradas     : {M['sujas_ignoradas_pct']:.1%}")
if df_cv is not None:
    print(f"\n(validação cruzada, {len(df_cv)} folds: acc teste {df_cv.acc_teste.mean() * 100:.2f}% ± {df_cv.acc_teste.std() * 100 if len(df_cv) > 1 else 0:.2f})")
print("\n" + classification_report(y_te, (p_te >= THRESHOLD).astype(int), target_names=CLASS_NAMES, digits=4))

fig, ax = plt.subplots(1, 5, figsize=(25, 4.3))
h = hist_final
ax[0].plot(h.epoca, h.loss_treino, "o-", ms=3, c="#c53030", label="loss treino")
ax[0].axhline(loss_te, c="#c53030", ls="--", label=f"loss teste ({loss_te:.3f})")
a0 = ax[0].twinx()
a0.plot(h.epoca, h.acc_treino * 100, "o-", ms=3, c="#2b6cb0", label="acc treino")
a0.axhline(acc_te_05 * 100, c="#2b6cb0", ls="--", label=f"acc teste ({acc_te_05 * 100:.1f}%)")
a0.set_ylim(0, 100); ax[0].set_xlabel("época"); ax[0].set_title("Treino final × teste")
h1, l1 = ax[0].get_legend_handles_labels(); h2, l2 = a0.get_legend_handles_labels(); ax[0].legend(h1 + h2, l1 + l2, fontsize=7)
cm = np.array(M["matriz_confusao"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
sns.heatmap(cmn, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1], cbar=False)
ax[1].set_xlabel("previsto"); ax[1].set_ylabel("real"); ax[1].set_title(f"Matriz de confusão (limiar {THRESHOLD:.2f})")
fpr, tpr, _ = roc_curve(y_te, p_te); ax[2].plot(fpr, tpr); ax[2].plot([0, 1], [0, 1], ":", c="gray")
ax[2].set_xlabel("limpas acionadas (FPR)"); ax[2].set_ylabel(f"recall {CLASS_NAMES[1]}"); ax[2].set_title(f"ROC · AUC {M['auc_roc']:.3f}"); ax[2].grid(alpha=0.3)
pr, rc, _ = precision_recall_curve(y_te, p_te); ax[3].plot(rc, pr)
ax[3].set_xlabel(f"recall {CLASS_NAMES[1]}"); ax[3].set_ylabel("precisão"); ax[3].set_title(f"Precisão × recall · AP {M['auc_pr']:.3f}"); ax[3].grid(alpha=0.3)
for c in range(2):
    ax[4].hist(p_te[y_te == c], bins=20, range=(0, 1), alpha=0.6, label=CLASS_NAMES[c])
ax[4].axvline(THRESHOLD, c="k", ls="--", label=f"limiar {THRESHOLD:.2f}"); ax[4].set_xlabel(f"P({CLASS_NAMES[1]})")
ax[4].set_title("Distribuição das probabilidades"); ax[4].legend()
plt.tight_layout(); plt.savefig(OUT / "metricas_teste.png", dpi=120); plt.show()

res = meta_te.copy()
res["p_limpar"] = p_te
res["previsto"] = np.where(p_te >= THRESHOLD, CLASS_NAMES[1], CLASS_NAMES[0])
res["erro"] = res.previsto != res.classe
display(res.groupby("fonte").agg(n=("erro", "size"), taxa_erro=("erro", "mean")).sort_values("n", ascending=False).round(3).head(15))

METRICAS = {"modelo": CFG["ARCH"], "img_size": S, "limiar": THRESHOLD,
            "treino": {"acc_ultima_epoca": hist_final.acc_treino.iloc[-1], "loss_ultima_epoca": hist_final.loss_treino.iloc[-1],
                       "acc_sem_aumento": acc_tr_eval, "loss_sem_aumento": loss_tr_eval, "n": int(len(y))},
            "teste": {"acc_corte_0_5": acc_te_05, "loss": loss_te, "no_limiar": M, "no_corte_0_5": M05, "ic95": IC},
            "validacao_cruzada": None if df_cv is None else df_cv.round(4).to_dict("records")}
(OUT / "metricas_modelo_final.json").write_text(json.dumps(METRICAS, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("salvo:", OUT / "metricas_modelo_final.json")

## 12. Onde o modelo final erra (Grad-CAM) e lista para revisão

Erros com confiança alta costumam ser foto na pasta errada ou sujeira invisível na resolução usada.
Saídas: `erros_teste.csv` e `revisar/real_<X>__prev_<Y>/`.

In [ ]:
def grad_cam(model, x, target=None):
    model.eval(); store = {}
    def hook(m, i, o):
        store["feat"] = o
        o.register_hook(lambda g: store.__setitem__("grad", g))
    hk = model.layer4.register_forward_hook(hook)
    out = model(x); c = out.argmax(1).item() if target is None else target
    model.zero_grad(); out[0, c].backward(); hk.remove()
    cam = torch.relu((store["grad"].mean((2, 3), keepdim=True) * store["feat"]).sum(1))[0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cv2.resize(cam.detach().cpu().numpy(), (S, S))

erros = res[res.erro].assign(confianca=lambda d: np.where(d.previsto == CLASS_NAMES[1], d.p_limpar, 1 - d.p_limpar))
erros = erros.sort_values("confianca", ascending=False)
print(f"{len(erros)} erros em {len(res)} imagens | com confiança > 0,9: {(erros.confianca > 0.9).sum()}")
top = erros.head(CFG["N_ERRORS_SHOW"])
if len(top):
    cols = 6; nrow = math.ceil(len(top) / cols)
    fig, axes = plt.subplots(2 * nrow, cols, figsize=(13, 5 * nrow), squeeze=False)
    for n, (i, r) in enumerate(top.iterrows()):
        rr, cc = 2 * (n // cols), n % cols
        img = X_te[i].permute(1, 2, 0).numpy()
        axes[rr, cc].imshow(img); axes[rr, cc].set_title(f"real {r.classe}\nprev {r.previsto} {r.confianca:.2f}", fontsize=7)
        axes[rr + 1, cc].imshow(img); axes[rr + 1, cc].imshow(grad_cam(final_model, to_input(X_te[i].unsqueeze(0))), cmap="jet", alpha=0.45)
        axes[rr + 1, cc].set_title(r.arquivo[:24], fontsize=6)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout(); plt.show()
erros[["arquivo", "classe", "previsto", "confianca", "fonte", "file"]].to_csv(OUT / "erros_teste.csv", index=False)
rev = OUT / "revisar"; shutil.rmtree(rev, ignore_errors=True)
for _, r in erros.iterrows():
    d = rev / f"real_{r.classe}__prev_{r.previsto}"; d.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.file, d / f"{r.confianca:.3f}_{r.arquivo}")
print("lista:", OUT / "erros_teste.csv", "| cópias em:", rev)

## 13. Exportação: ONNX + labels.json + model_card.json

**Contrato do `model.onnx`** (normalização ImageNet e softmax embutidos):
- entrada `input`: `float32` NHWC `[N, 224, 224, 3]`, RGB 0..255, **já tratada** pelo `preprocess.py` do notebook de tratamento;
- saída `probs`: `[N, 2]` = `[P(NaoLimpar), P(Limpar)]`;
- decisão: `Limpar` se `P(Limpar) >= model_card["decisao"]["limiar"]`.

```python
import json, onnxruntime as ort
from preprocess import preprocess            # o do notebook de tratamento
card = json.load(open("model_card.json"))
sess = ort.InferenceSession("model.onnx", providers=["CPUExecutionProvider"])
p = sess.run(None, {"input": preprocess("foto.jpg")})[0][0]
classe = "Limpar" if p[1] >= card["decisao"]["limiar"] else "NaoLimpar"
```

In [ ]:
import importlib, importlib.util, subprocess

def ensure(pkgs):
    falta = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if falta:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *falta], check=True)
        importlib.invalidate_caches()

class ExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(MEAN).view(1, 3, 1, 1) * 255.0)
        self.register_buffer("std", torch.tensor(STD).view(1, 3, 1, 1) * 255.0)
    def forward(self, x):
        return torch.softmax(self.model((x.permute(0, 3, 1, 2) - self.mean) / self.std), dim=1)

(OUT / "labels.json").write_text(json.dumps(CLASS_NAMES, ensure_ascii=False), encoding="utf-8")
onnx_info = None
if CFG["EXPORT_ONNX"]:
    ensure([("onnx", "onnx"), ("onnxruntime", "onnxruntime")])
    import onnx, onnxruntime as ort
    wrapper = ExportWrapper(copy.deepcopy(final_model).float().cpu().eval()).eval()
    xs = X_te[:32].permute(0, 2, 3, 1).numpy().astype("float32")
    onnx_path = OUT / "model.onnx"; onnx_path.unlink(missing_ok=True)
    kw = dict(input_names=["input"], output_names=["probs"], opset_version=CFG["ONNX_OPSET"])
    metodo, falhas = None, []
    try:
        try:
            torch.onnx.export(wrapper, (torch.from_numpy(xs[:2]),), str(onnx_path), dynamo=False,
                              dynamic_axes={"input": {0: "batch"}, "probs": {0: "batch"}}, **kw)
        except TypeError:
            torch.onnx.export(wrapper, (torch.from_numpy(xs[:2]),), str(onnx_path),
                              dynamic_axes={"input": {0: "batch"}, "probs": {0: "batch"}}, **kw)
        metodo = "torch.onnx.export (TorchScript)"
    except Exception as e:
        falhas.append(f"TorchScript: {type(e).__name__}: {e}")
    if metodo is None:
        try:
            ensure([("onnxscript", "onnxscript")])
            torch.onnx.export(wrapper, (torch.from_numpy(xs[:2]),), str(onnx_path), dynamo=True,
                              dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=1024)}}, **kw)
            metodo = "torch.onnx.export (dynamo)"
        except Exception as e:
            falhas.append(f"dynamo: {type(e).__name__}: {e}")
    if metodo is None:
        raise RuntimeError("falha na exportação ONNX:\n  - " + "\n  - ".join(falhas))
    onnx.checker.check_model(str(onnx_path))
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    p_onnx = np.concatenate([sess.run(None, {"input": xs[i:i + 8]})[0] for i in range(0, len(xs), 8)])
    with torch.no_grad():
        diff = float(np.abs(p_onnx - wrapper(torch.from_numpy(xs)).numpy()).max())
    print(f"ONNX via {metodo}: {onnx_path.stat().st_size / 1e6:.1f} MB | diferença máxima ONNX x PyTorch: {diff:.2e} "
          f"({'ok' if diff < 1e-3 else 'ATENÇÃO: diferença alta'})")
    onnx_info = {"arquivo": "model.onnx", "opset": CFG["ONNX_OPSET"], "exportado_via": metodo,
                 "bytes": onnx_path.stat().st_size, "max_abs_diff_vs_pytorch": diff,
                 "input": {"name": "input", "shape": ["batch", S, S, 3], "dtype": "float32", "layout": "NHWC",
                           "cor": "RGB", "faixa": [0, 255], "observacao": "imagem já tratada pelo preprocess.py"},
                 "output": {"name": "probs", "shape": ["batch", 2], "significado": [f"P({c})" for c in CLASS_NAMES]}}

card = {
    "nome": "classificador de sujeira em placas solares",
    "versao": f"{CFG['ARCH']}-{S}-{dt.date.today():%Y%m%d}",
    "criado_em": dt.datetime.now().isoformat(timespec="seconds"),
    "arquitetura": CFG["ARCH"], "pretreino": "ImageNet (torchvision)" if CFG["PRETRAINED"] else None,
    "origem_dos_pesos": "treino final do zero com 100% do dataset de treino",
    "labels": CLASS_NAMES, "classe_positiva": CLASS_NAMES[1],
    "arquivos": {"onnx": "model.onnx" if onnx_info else None, "pytorch": "modelo_placas.pth",
                 "labels": "labels.json", "metricas": "metricas_modelo_final.json"},
    "onnx": onnx_info,
    "preprocessamento": {"descricao": "use o preprocess.py do notebook de tratamento (enquadramento + 224x224 RGB)",
                         "img_size": S},
    "decisao": {"limiar": THRESHOLD, "criterio": CFG["THRESHOLD_MODE"],
                "regra": f"{CLASS_NAMES[1]} se P({CLASS_NAMES[1]}) >= limiar, senão {CLASS_NAMES[0]}"},
    "treino": {k: CFG[k] for k in ("EPOCHS", "BATCH", "LR", "WEIGHT_DECAY", "LABEL_SMOOTHING", "DROPOUT",
                                    "FREEZE_EPOCHS", "AUG_ROT90", "AUG_RANDOM_DEG", "AUG_FLIP", "COLOR_JITTER", "SEED")},
    "dados": {"dataset": Path(CFG["TRAIN_PATH"]).name,
              "teste": Path(CFG["TEST_PATH"]).name if CFG["TEST_PATH"] else f"split interno ({CFG['TEST_FRAC']:.0%} das fotos reais, por foto original)",
              "n_treino": int(len(y)), "n_teste": int(len(y_te)), "sinteticas_no_treino": int(meta.sintetica.sum())},
    "avaliacao_teste": {k: M[k] for k in ("n", "acuracia", "acuracia_balanceada", "f1_macro", "auc_roc", "auc_pr",
                                          "limpas_que_acionariam_pct", "sujas_ignoradas_pct", "matriz_confusao")}
                       | {"loss": loss_te, "ic95": IC},
}
(OUT / "model_card.json").write_text(json.dumps(card, indent=2, ensure_ascii=False, default=float), encoding="utf-8")

pacote = OUT / "modelo_placas.zip"
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
    for nome in ["model.onnx", "labels.json", "model_card.json", "metricas_modelo_final.json", "metricas_teste.png", "modelo_placas.pth"]:
        if (OUT / nome).exists():
            z.write(OUT / nome, nome)
print("pacote:", pacote, f"({pacote.stat().st_size / 1e6:.1f} MB) — junte o preprocess.py do notebook de tratamento")
if IN_COLAB:
    from google.colab import files
    files.download(str(pacote))

## 14. Observações

1. **Comece rápido:** `QUICK=True` confere que tudo roda; depois `N_FOLDS_TO_RUN=1` ou `2` para uma primeira estimativa.
   Com `N_FOLDS_TO_RUN=0`, o notebook vai direto ao treino final (limiar fixo em 0,5).
2. **Treino × teste:** se a acurácia de treino passar de ~98% e a de teste ficar bem abaixo, é overfitting: reduza
   `EPOCHS`, aumente `DROPOUT` ou use `resnet18`.
3. **Teste:** com o split interno, a divisão é por foto original e fica em `particao.csv` (rode de novo com o mesmo
   `SEED` para obter a mesma divisão). Com um teste externo, ele precisa ter passado pelo mesmo tratamento.
4. **Limiar:** para manutenção, deixar passar placa suja costuma custar mais que um alarme falso; nesse caso use
   `THRESHOLD_MODE="recall"` com `RECALL_TARGET` alto.